# SSVEP FB CCA Classifier Test with Synthetic EEG Data

This notebook demonstrates how to test the **SSVEP FB CCA Classifier** using synthetic EEG data consisting of sine waves at specific frequencies.

## Steps:
1. Generate synthetic SSVEP data using sine waves at specific target frequencies.
2. Set up the **SSVEP FB CCA Classifier** with appropriate parameters.
3. Use the classifier to predict based on the generated synthetic data.
4. Visualize the synthetic signals.

### Required Libraries

In [16]:
import numpy as np
import logging
import re
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import CCA
from scipy.signal import chirp, find_peaks
from io import StringIO
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import clear_output
from bci_essentials.classification.ssvep_fbcca_classifier import (
    SsvepFbCcaClassifier,
)

### Step 1: Generate Synthetic SSVEP Data at a specific frequency

In [17]:
def generate_synthetic_ssvep_data(frequency, fsample, n_samples, n_trials=1, noise_level=0.1, harmonics=[2, 3]):
    """
    Generates synthetic SSVEP data with optional noise and harmonics.

    Parameters:
    - frequency: The base frequency of the sine wave.
    - fsample: The sampling frequency.
    - n_samples: The number of samples per trial.
    - n_trials: The number of trials (default is 1).
    - noise_level: The standard deviation of Gaussian noise (default is 0.1).
    - harmonics: A list of harmonic multiples to add to the base frequency (default is [2, 3]).

    Returns:
    - data: A NumPy array of shape (n_trials, 1, n_samples) containing the synthetic SSVEP data.
    """
    # Ensure the frequency is a float
    frequency = float(frequency)  # Convert to float if it's not already
    
    # Generate time vector
    t = np.linspace(0, n_samples / fsample, n_samples)
    
    # Initialize data array
    data = np.zeros((n_trials, 1, n_samples))  # n_trials x 1 x n_samples (for a single frequency)
    
    for trial in range(n_trials):
        # Generate the base sine wave
        signal = np.sin(2 * np.pi * frequency * t)
        
        # Add harmonics
        for harmonic in harmonics:
            signal += (1 / harmonic) * np.sin(2 * np.pi * harmonic * frequency * t)
        
        # Add Gaussian noise
        noise = np.random.normal(0, noise_level, n_samples)
        signal += noise
        
        # Store the signal in the data array
        data[trial, 0, :] = signal
    
    return data


### Step 2: Set Parameters and Generate Data


In [ ]:
# Set parameters
fsample = 256  # Sampling frequency (Hz)
n_samples = 5 * 256  # Number of samples per trial
n_trials = 10  # Number of trials to simulate
noise_level = 0.4 # Noise level for the synthetic data

# Generate synthetic SSVEP data (sine waves at specific frequencies)
synthetic_7 = generate_synthetic_ssvep_data(7.69, fsample, n_samples, n_trials, noise_level)
synthetic_10 = generate_synthetic_ssvep_data(10, fsample, n_samples, n_trials, noise_level)
synthetic_12 = generate_synthetic_ssvep_data(12.5, fsample, n_samples, n_trials, noise_level)
synthetic_14 = generate_synthetic_ssvep_data(14.28, fsample, n_samples, n_trials, noise_level)

# Combine the synthetic data into a single array, with trials from each frequency in random order
synthetic_data = np.concatenate((synthetic_7, synthetic_10, synthetic_12, synthetic_14), axis=0)

# Create labels for the synthetic data
labels = np.concatenate((np.full(n_trials, 0), np.full(n_trials, 1), np.full(n_trials, 2), np.full(n_trials, 3)))

# Shuffle the data and labels together
indices = np.arange(synthetic_data.shape[0])
np.random.shuffle(indices)
synthetic_data = synthetic_data[indices]
labels = labels[indices]

# Reshape the data to match the expected input shape for the classifier
synthetic_data = synthetic_data.reshape(synthetic_data.shape[0], 1, -1)

print("Synthetic data shape:", synthetic_data.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels)

Synthetic data shape: (40, 1, 1280)
Labels shape: (40,)
Labels: [2 2 3 3 0 1 2 0 2 3 0 2 0 1 2 2 0 1 3 1 0 3 3 1 1 3 1 0 2 0 1 2 3 0 0 3 3
 1 2 1]


### Step 3: Set up the Classifier


In [19]:
classifier = SsvepFbCcaClassifier(subset = [])

# Set up SSVEP settings (use default filter bank or customize)
target_frequencies = np.array([7.692307, 10, 12.5, 14.28571])
n_harmonics = 3  # Number of harmonics to use for SSVEP classification
fb_cutoffs = np.array([[i, 27] for i in range(3, 25, 2)])   # Filter bank cut-off frequencies
filter_order = 6
concatenate_trials = True  # Concatenate trials for training

classifier.set_ssvep_settings(
    fsample=256.0,
    n_harmonics=n_harmonics,  # Number of harmonics to use for SSVEP classification
    target_freqs=target_frequencies,
    n_samples=n_samples,
    filter_bank=fb_cutoffs,
    concatenate_trials=concatenate_trials,
)    

2025-04-09 15:48:49 - INFO - bci_essentials.classification.generic_classifier : Initializing the classifier


### Step 4: Prediction


In [20]:
true = labels

# Create a string buffer to capture log output
log_stream = StringIO()
handler = logging.StreamHandler(log_stream)
logger = logging.getLogger()
logger.addHandler(handler)

# Predict with the classifier for each trial
pred = []
for i in range(synthetic_data.shape[0]):
    prediction = classifier.predict(synthetic_data[i:i+1])

# Get log output as string
log_output = log_stream.getvalue()

for line in log_output.split('\n'):
    if "Predicted label: " in line:
        pred_label = int(re.search(r"Predicted label: (\d+)", line).group(1))
        pred.append(pred_label)

clear_output()

# Calculate and display accuracy
accuracy = np.mean(true == pred)
print(f"Accuracy: {accuracy:.2%}")

# Create confusion matrix
conf_matrix = confusion_matrix(true, pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(true, pred))
print(f"Number of samples: {len(true)}")
print(f"True labels: {true}")
print(f"Predicted labels: {pred}")


Accuracy: 100.00%

Confusion Matrix:
[[10  0  0  0]
 [ 0 10  0  0]
 [ 0  0 10  0]
 [ 0  0  0 10]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Number of samples: 40
True labels: [2 2 3 3 0 1 2 0 2 3 0 2 0 1 2 2 0 1 3 1 0 3 3 1 1 3 1 0 2 0 1 2 3 0 0 3 3
 1 2 1]
Predicted labels: [2, 2, 3, 3, 0, 1, 2, 0, 2, 3, 0, 2, 0, 1, 2, 2, 0, 1, 3, 1, 0, 3, 3, 1, 1, 3, 1, 0, 2, 0, 1, 2, 3, 0, 0, 3, 3, 1, 2, 1]
